In [1]:
import pandas as pd
import readability
import os
import numpy as np
import random
import torch
from bert_score import BERTScorer
import re
from nltk.translate import meteor
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
import math

In [2]:
from transformers import logging
logging.set_verbosity_error() # make sure only important transformers logging output is visible

In [3]:
OUTPUT_DIR = 'outputScores/'

In [4]:
generation_types = ['MATAVE', 'LDA', 'allTopicModels']
input_prompt_paths = ['./dataGeneration/fake_notes.xlsx', './dataGeneration/fake_notes.xlsx', './dataGeneration/fake_notes.xlsx']
input_generated_data_dirs = ['dataGeneration/processedData/MATAVE.csv', 'dataGeneration/processedData/LDA.csv', 'dataGeneration/processedData/allTopicModels.csv']

In [5]:
# Only need to run once.
# nltk.download('all')

In [6]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [7]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s/]', '', text)
    # tokenize
    tokens = word_tokenize(text.lower())
    # remove stop words
    filtered_tokens = [token for token in tokens if token not in stopwords.words('english')]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    # join the tokens back into a string
    processed_text = ' '.join(lemmatized_tokens)
    return processed_text

In [8]:
prompt_df = pd.read_excel(input_prompt_paths[0])
temp_df = pd.read_csv(f"./{input_generated_data_dirs[0]}")

In [9]:
def make_score_df(generation_type, input_prompt_path, input_generated_data):
    prompt_df = pd.read_excel(input_prompt_path)
    average_scores_dict = []
    # Make BERT scorer.
    scorer = BERTScorer(model_type="bert-base-uncased")

    topic_model_df = pd.read_csv(f"./{input_generated_data}")
    for temp_prompt in prompt_df.iterrows():
        temp_df = topic_model_df[topic_model_df['prompt_id'] == f"{temp_prompt[0] % 5}_{temp_prompt[1]['Needs']}"]
        # --------- Readability (number linked to grade of reading level)s ---------
        references = []
        candidates = []
        meteors = []
        sentiments = []
        subjectivities = []
        abs_sentiment_diffs = []
        abs_subjectivity_diffs = []
        # Make readability metric.
        preprocessed_prompt = preprocess_text(temp_prompt[1]['Note'])
        reference_blob = TextBlob(preprocessed_prompt)
        for temp_generation in temp_df['report'].tolist():
            processed_generation = preprocess_text(temp_generation)
            candidate_blob = TextBlob(processed_generation)
            if candidate_blob.sentences and reference_blob.sentences:
                # Make candidates and references without punctuation for metrics (BERTScore penalizes punctuation, we should take this into account - https://aclanthology.org/2023.findings-acl.381.pdf).
                reference = re.sub(r'[^\w\s/]', '', temp_prompt[1]['Note'])
                candidate = re.sub(r'[^\w\s/]', '', temp_generation)
                references.append(reference)
                candidates.append(candidate)
                sentiments.append(candidate_blob.sentences[0].sentiment.polarity)
                subjectivities.append(candidate_blob.sentences[0].sentiment.subjectivity)
                # Make METEOR
                meteors.append(meteor([word_tokenize(candidate)], word_tokenize(reference)))
                # Make sentiment and subjectivity absolute differences.                    
                abs_sentiment_diffs.append(math.sqrt((reference_blob.sentences[0].sentiment.polarity - candidate_blob.sentences[0].sentiment.polarity) ** 2))
                abs_subjectivity_diffs.append(math.sqrt((reference_blob.sentences[0].sentiment.subjectivity - candidate_blob.sentences[0].sentiment.subjectivity) ** 2))

        # Make BERTScore
        _, _, F1 = scorer.score(candidates, references)

        average_scores_dict.append({
            'topic_model': input_generated_data.replace('.csv', '').replace('dataGeneration/processedData/', ''), 
            'needs': temp_prompt[1]['Needs'],
            'prompt_id': f"{temp_prompt[0] % 5}_{temp_prompt[1]['Needs']}",
            'generation_type': generation_type,
            'number_of_notes': len(temp_df),
            # save the F1 values, as these are recommended for use by the BERTScore authors - http://arxiv.org/abs/1904.09675
            'bertscore': float(F1.mean()),
            'meteor': sum(meteors) / len(meteors),
            'sentiment': sum(sentiments) / len(sentiments),
            'subjectivity': sum(subjectivities) / len(subjectivities),
            'abs_sentiment_diff': sum(abs_sentiment_diffs) / len(abs_sentiment_diffs),
            'abs_subjectivity_diff': sum(abs_subjectivity_diffs) / len(abs_subjectivity_diffs)})
                
    # min-max normalize every score
    resultant_df = pd.DataFrame(average_scores_dict)
    return resultant_df


In [10]:
all_dfs = []
for generation_type, input_prompt_path, input_generated_data_dir in zip(generation_types, input_prompt_paths, input_generated_data_dirs):
    all_dfs.append(make_score_df(generation_type, input_prompt_path, input_generated_data_dir))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
combined_df = pd.concat(all_dfs)

In [12]:
scaled_df = combined_df.copy()

metric_cols = [
    "bertscore",
    "meteor",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

# Min-max normalize safely.
for col in metric_cols:
    min_val = scaled_df[col].min()
    max_val = scaled_df[col].max()

    if max_val - min_val == 0:
        scaled_df[col] = 0.0
    else:
        scaled_df[col] = (scaled_df[col] - min_val) / (max_val - min_val)

# Invert the lower is better columns.
lower_is_better = [
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]

for col in lower_is_better:
    scaled_df[col] = 1 - scaled_df[col]

# Get overall scores.
scaled_df["overall_score"] = scaled_df[[
    "bertscore",
    "meteor",
    "abs_sentiment_diff",
    "abs_subjectivity_diff"
]].mean(axis=1)

# Get final summary table.
summary = (
    scaled_df
    .groupby(["generation_type"])["overall_score"]
    .mean()
    .reset_index()
    .sort_values(["overall_score"], ascending=False)
)

# Save outputs. 
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok=True)
scaled_df.to_csv(f'./{OUTPUT_DIR}/evaluated_scaled_results.csv', index=False)
combined_df.to_csv(f'./{OUTPUT_DIR}/raw_evaluation_results.csv', index=False)
summary.to_csv(f'./{OUTPUT_DIR}/dataset_summary.csv', index=False)

In [13]:
print("------ Summary Performance ------")
summary

------ Summary Performance ------


,generation_type,overall_score
0,LDA,0.572163
2,allTopicModels,0.543216
1,MATAVE,0.520031


In [14]:
print("------ Scaled Scores ------")
scaled_df.sort_values(by='overall_score', ascending=False)

------ Scaled Scores ------


,topic_model,needs,prompt_id,generation_type,number_of_notes,bertscore,meteor,sentiment,subjectivity,abs_sentiment_diff,abs_subjectivity_diff,overall_score
4,LDA,met,4_met,LDA,54,1.000000,1.000000,0.087165,0.323865,0.819052,0.612865,0.857979
6,LDA,unmet,1_unmet,LDA,273,0.751211,0.428225,-0.004653,0.387166,1.000000,0.886219,0.766414
9,LDA,unmet,4_unmet,LDA,254,0.604792,0.925352,0.040565,0.391242,0.980953,0.487099,0.749549
6,allTopicModels,unmet,1_unmet,allTopicModels,1014,0.673612,0.378648,0.012804,0.394681,0.982958,0.901565,0.734196
0,LDA,met,0_met,LDA,169,0.372654,0.664079,0.128922,0.458262,0.887393,1.000000,0.731031
4,allTopicModels,met,4_met,allTopicModels,250,0.807942,0.800914,0.111927,0.345708,0.754416,0.554232,0.729376
9,allTopicModels,unmet,4_unmet,allTopicModels,1064,0.547492,0.929049,0.091149,0.388288,0.882009,0.488734,0.711821
0,allTopicModels,met,0_met,allTopicModels,770,0.384333,0.640730,0.159099,0.462008,0.820962,0.980939,0.706741
6,MATAVE,unmet,1_unmet,MATAVE,234,0.583080,0.320807,0.033171,0.403448,0.963076,0.919469,0.696608
0,MATAVE,met,0_met,MATAVE,216,0.393471,0.622461,0.182710,0.464939,0.768986,0.966026,0.687736


In [15]:
print("------ Raw Dataframe ------")
combined_df

------ Raw Dataframe ------


,topic_model,needs,prompt_id,generation_type,number_of_notes,bertscore,meteor,sentiment,subjectivity,abs_sentiment_diff,abs_subjectivity_diff
0,MATAVE,met,0_met,MATAVE,216,0.536977,0.186625,0.182710,0.464939,0.194684,0.132647
1,MATAVE,met,1_met,MATAVE,239,0.517871,0.148615,0.252973,0.492244,0.217416,0.167495
2,MATAVE,met,2_met,MATAVE,200,0.521462,0.114051,0.213108,0.446913,0.149747,0.221264
3,MATAVE,met,3_met,MATAVE,312,0.505836,0.143831,0.231424,0.473888,0.202451,0.268317
4,MATAVE,met,4_met,MATAVE,71,0.558219,0.190471,0.130760,0.362321,0.216042,0.310172
5,MATAVE,unmet,0_unmet,MATAVE,243,0.511789,0.123360,0.085863,0.343852,0.369875,0.508411
6,MATAVE,unmet,1_unmet,MATAVE,234,0.551983,0.143710,0.033171,0.403448,0.129640,0.150756
7,MATAVE,unmet,2_unmet,MATAVE,249,0.527464,0.121268,0.100101,0.386183,0.203036,0.172768
8,MATAVE,unmet,3_unmet,MATAVE,271,0.523171,0.130398,0.056987,0.367809,0.131581,0.367809
9,MATAVE,unmet,4_unmet,MATAVE,278,0.545023,0.230722,0.137366,0.385588,0.187104,0.317722


In [16]:
# Global model comparison. 
global_summary = (
    scaled_df
    .groupby("generation_type")[[
        "bertscore",
        "meteor",
        "abs_sentiment_diff",
        "abs_subjectivity_diff",
        "overall_score"
    ]]
    .mean()
    .sort_values("overall_score", ascending=False)
)

In [17]:
print("\n------ Global Model Performance (Scaled) ------\n")
global_summary


------ Global Model Performance (Scaled) ------



,bertscore,meteor,abs_sentiment_diff,abs_subjectivity_diff,overall_score
generation_type,,,,,
LDA,0.407811,0.438797,0.788524,0.653520,0.572163
allTopicModels,0.353092,0.410150,0.766863,0.642759,0.543216
MATAVE,0.305058,0.388251,0.752655,0.634159,0.520031
